# Day 08: The Training Loop — Loss, Optimizer, Learning Rate

**Goal:** Master the building blocks of every training loop. After today, you'll have a reusable trainer that works for ANY model.

### What we'll cover:

1. **Loss functions** — MSE, BCE, Cross-Entropy (which to use when)
2. **Optimizers** — SGD vs Adam vs AdamW (visual comparison)
3. **Learning rate scheduling** — cosine, warmup (what LLMs use)
4. **Mini-batching** — why we don't use the whole dataset at once
5. **Production training loop** — `model.train()`, `model.eval()`, device handling
6. **Checkpointing** — saving and loading models
7. **Early stopping** — when to call it done

By the end, you'll have a `Trainer` class that handles all of this for you.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

## 1. Loss Functions — Pick the Right One

Different tasks need different loss functions. Let's see all 3 in action:

In [ ]:
# Three different problems, three different loss functions

# REGRESSION: predict a number (e.g., house price)
y_pred_reg = torch.tensor([3.5, 7.2, 1.8])
y_true_reg = torch.tensor([3.0, 7.0, 2.0])
mse = F.mse_loss(y_pred_reg, y_true_reg)
print(f"Regression task → MSE loss: {mse:.4f}")

# BINARY CLASSIFICATION: yes/no (e.g., spam detection)
# Note: use logits (raw scores) not probabilities, more numerically stable
logits_bin = torch.tensor([2.0, -1.5, 0.5])    # raw scores
y_true_bin = torch.tensor([1.0, 0.0, 1.0])    # 0/1 labels
bce = F.binary_cross_entropy_with_logits(logits_bin, y_true_bin)
print(f"Binary class. → BCE loss:  {bce:.4f}")

# MULTI-CLASS CLASSIFICATION: pick one of K (e.g., next token in vocab)
# Logits shape: (batch, num_classes)
# Targets: integers (the correct class index)
logits_multi = torch.tensor([
    [0.1, 2.5, 0.2, 1.0],    # sample 0: predicts class 1
    [3.0, 0.5, 0.1, 0.2],    # sample 1: predicts class 0
    [0.2, 0.1, 1.8, 0.5],    # sample 2: predicts class 2
])
y_true_multi = torch.tensor([1, 0, 2])  # correct classes
ce = F.cross_entropy(logits_multi, y_true_multi)
print(f"Multi-class   → CE loss:   {ce:.4f}")
print(f"               (LLMs use this — 'next token' is multi-class!)")

### Why "with_logits" versions exist

Notice we used `binary_cross_entropy_WITH_LOGITS` instead of plain `binary_cross_entropy`. Here's why:

```
Standard:    loss(sigmoid(z), y)    ← sigmoid then BCE separately
With logits: loss_with_logits(z, y) ← combined for numerical stability
```

When `z` is a huge number, `sigmoid(z)` rounds to 1.0, then `log(1 - 1.0) = -inf` → NaN! The "with_logits" version does the math more carefully and avoids this.

**Rule:** Always prefer the `_with_logits` version when available.

---

## 2. Optimizers — How Weights Get Updated

Let's compare the most common optimizers on a tricky loss function (one with valleys):

In [ ]:
# Compare optimizers on a "narrow valley" loss landscape
# (this is where SGD struggles but Adam shines)

def loss_function(x, y):
    """A loss with a long thin valley — tricky for naive SGD."""
    return x**2 + 100 * y**2  # the y² term has 100x bigger gradient

def run_optimizer(opt_class, lr, **kwargs):
    """Run a given optimizer for 50 steps and return the path."""
    pos = torch.tensor([5.0, 0.5], requires_grad=True)
    optimizer = opt_class([pos], lr=lr, **kwargs)
    
    path = [pos.detach().clone().tolist()]
    for _ in range(50):
        loss = loss_function(pos[0], pos[1])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        path.append(pos.detach().clone().tolist())
    return np.array(path)

# Run each optimizer
path_sgd       = run_optimizer(torch.optim.SGD, lr=0.001)
path_momentum  = run_optimizer(torch.optim.SGD, lr=0.001, momentum=0.9)
path_adam      = run_optimizer(torch.optim.Adam, lr=0.5)

# Plot the loss landscape with each optimizer's path
x_grid = np.linspace(-1, 6, 100)
y_grid = np.linspace(-1, 1, 100)
X, Y = np.meshgrid(x_grid, y_grid)
Z = X**2 + 100 * Y**2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, path, name in zip(axes, [path_sgd, path_momentum, path_adam],
                            ['SGD', 'SGD + Momentum', 'Adam']):
    ax.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.5)
    ax.plot(path[:, 0], path[:, 1], 'r-o', markersize=3, linewidth=1.5)
    ax.plot(path[0, 0], path[0, 1], 'gs', markersize=12, label='Start')
    ax.plot(path[-1, 0], path[-1, 1], 'r*', markersize=15, label='End')
    final_loss = loss_function(torch.tensor(path[-1, 0]), torch.tensor(path[-1, 1])).item()
    ax.set_title(f'{name}\nfinal loss: {final_loss:.4f}')
    ax.set_xlabel('w1')
    ax.set_ylabel('w2')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print("  SGD: zigzags, slow to converge — the y² gradient is too strong")
print("  SGD+Momentum: smoother, but still some oscillation")
print("  Adam: adapts per-parameter, gets to the minimum efficiently")

## 3. Learning Rate Scheduling

The learning rate often needs to CHANGE during training:
- **Start high** to make fast progress
- **Decrease over time** so you don't overshoot the minimum
- **Sometimes warm up** at the very beginning

Let's visualize common schedules:

In [ ]:
# Visualize different learning rate schedules

def get_lrs(scheduler_factory, total_epochs=100):
    """Run a scheduler and return the LR at each epoch."""
    dummy = nn.Linear(1, 1)
    optimizer = torch.optim.SGD(dummy.parameters(), lr=0.1)
    scheduler = scheduler_factory(optimizer)
    
    lrs = []
    for _ in range(total_epochs):
        lrs.append(optimizer.param_groups[0]['lr'])
        scheduler.step()
    return lrs

# Constant (no schedule)
lrs_constant = [0.1] * 100

# Step decay — drop by 10x every 30 epochs
lrs_step = get_lrs(lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=30, gamma=0.1))

# Exponential decay
lrs_exp = get_lrs(lambda opt: torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.97))

# Cosine annealing — smooth decrease
lrs_cosine = get_lrs(lambda opt: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100))

# Warmup + cosine (LLM standard)
def warmup_cosine_lambda(epoch, warmup=10, total=100):
    if epoch < warmup:
        return epoch / warmup
    progress = (epoch - warmup) / (total - warmup)
    return 0.5 * (1 + np.cos(np.pi * progress))

lrs_warmup = get_lrs(lambda opt: torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=warmup_cosine_lambda))

plt.figure(figsize=(11, 5))
plt.plot(lrs_constant, label='Constant (no schedule)', linewidth=2)
plt.plot(lrs_step, label='Step decay (every 30 epochs)', linewidth=2)
plt.plot(lrs_exp, label='Exponential decay', linewidth=2)
plt.plot(lrs_cosine, label='Cosine annealing', linewidth=2)
plt.plot(lrs_warmup, label='Warmup + cosine (LLM standard)', linewidth=2, linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Learning rate')
plt.title('Learning Rate Schedules')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("LLM training (GPT, Llama, etc.) typically uses warmup + cosine.")
print("Why warmup? Random initial weights mean huge gradients early on.")
print("           Slowly increasing LR avoids destabilizing the model.")

## 4. Mini-Batching — One Update Per Batch

Until now we did ONE update per epoch (using all data at once). Real training does ONE update per BATCH (small chunk).

Why? Three reasons:
1. **Memory** — can't fit a million samples in GPU memory at once
2. **More updates** — 1000 updates with batches > 1 update with full data, in same time
3. **Noise helps** — random batches → noisy gradient → escapes local minima

Let's compare:

In [ ]:
# Generate a small classification dataset
torch.manual_seed(42)
n_samples = 500
X_data = torch.randn(n_samples, 2)
y_data = (X_data[:, 0] + X_data[:, 1] > 0).float().unsqueeze(1)  # simple decision rule

# Split train/val
n_train = 400
X_train, y_train = X_data[:n_train], y_data[:n_train]
X_val, y_val = X_data[n_train:], y_data[n_train:]

# Wrap in PyTorch Dataset + DataLoader for batching
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

# DataLoader: yields batches one at a time
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Total samples: {n_samples}")
print(f"Train: {n_train}, Val: {n_samples - n_train}")
print(f"Batch size: 32")
print(f"Batches per epoch: {len(train_loader)}")
print(f"\nThis means: {len(train_loader)} weight updates per epoch (not just 1!)")

## 5. The Full Production Training Loop

Now let's combine EVERYTHING — proper batching, train/val tracking, learning rate scheduling, model.train()/eval() modes, and device handling:

In [ ]:
# A simple model for our binary classification task
class SimpleNet(nn.Module):
    def __init__(self, input_dim=2, hidden=16, output_dim=1):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, output_dim)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)   # return logits (no sigmoid here)


# THE PRODUCTION TRAINING LOOP

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    """Run one epoch of training. Returns average loss."""
    model.train()                          # enables dropout, batchnorm training mode
    total_loss = 0
    n_samples = 0
    
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)        # move to GPU/CPU
        batch_y = batch_y.to(device)
        
        # The 5-line core
        logits = model(batch_x)
        loss = loss_fn(logits, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(batch_x)
        n_samples += len(batch_x)
    
    return total_loss / n_samples


def evaluate(model, loader, loss_fn, device):
    """Evaluate on a dataset. Returns avg loss + accuracy."""
    model.eval()                            # disables dropout, switches batchnorm
    total_loss = 0
    correct = 0
    n_samples = 0
    
    with torch.no_grad():                  # no gradient tracking → faster, less memory
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            logits = model(batch_x)
            loss = loss_fn(logits, batch_y)
            
            # Predictions: sigmoid(logits) > 0.5 → class 1
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == batch_y).sum().item()
            
            total_loss += loss.item() * len(batch_x)
            n_samples += len(batch_x)
    
    return total_loss / n_samples, correct / n_samples


# Setup
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device}")

torch.manual_seed(42)
model = SimpleNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
loss_fn = nn.BCEWithLogitsLoss()

# Track losses and accuracies
train_losses, val_losses, val_accs, lrs = [], [], [], []

for epoch in range(50):
    # Get current LR before stepping
    lrs.append(optimizer.param_groups[0]['lr'])
    
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
    scheduler.step()
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    if epoch % 10 == 0 or epoch == 49:
        print(f"Epoch {epoch:3d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.3f}, lr={lrs[-1]:.5f}")

In [ ]:
# Visualize: loss curves, accuracy, and the LR schedule

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Loss curves
axes[0].plot(train_losses, 'b-', label='Train loss', linewidth=2)
axes[0].plot(val_losses, 'r-', label='Val loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(val_accs, 'g-', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title(f'Validation Accuracy (final: {val_accs[-1]*100:.1f}%)')
axes[1].set_ylim(0.5, 1.0)
axes[1].grid(True, alpha=0.3)

# LR schedule
axes[2].plot(lrs, 'm-', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning rate')
axes[2].set_title('Cosine LR schedule')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final validation accuracy: {val_accs[-1]*100:.1f}%")

## 6. Checkpointing — Save Your Work

After training for hours, you don't want to lose progress! Let's save and load the model:

In [ ]:
import os

# Save the model + optimizer state in a single checkpoint
checkpoint = {
    'epoch': 50,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'val_loss': val_losses[-1],
    'val_acc': val_accs[-1],
}

ckpt_path = '/tmp/day08_checkpoint.pth'
torch.save(checkpoint, ckpt_path)
print(f"Saved checkpoint to {ckpt_path}")
print(f"  File size: {os.path.getsize(ckpt_path) / 1024:.1f} KB")

# --- Now let's load it into a fresh model ---
new_model = SimpleNet().to(device)
new_optimizer = torch.optim.Adam(new_model.parameters(), lr=0.01)

print("\nFresh model BEFORE loading:")
fresh_loss, fresh_acc = evaluate(new_model, val_loader, loss_fn, device)
print(f"  val_loss={fresh_loss:.4f}, val_acc={fresh_acc:.3f}  (random predictions)")

# Load the checkpoint
loaded = torch.load(ckpt_path)
new_model.load_state_dict(loaded['model_state_dict'])
new_optimizer.load_state_dict(loaded['optimizer_state_dict'])

print("\nFresh model AFTER loading checkpoint:")
loaded_loss, loaded_acc = evaluate(new_model, val_loader, loss_fn, device)
print(f"  val_loss={loaded_loss:.4f}, val_acc={loaded_acc:.3f}  (matches trained model!)")

print(f"\nSaved at epoch {loaded['epoch']} with val_acc={loaded['val_acc']:.3f}")

## 7. Early Stopping + Best Model Tracking

Stop training when val loss stops improving, and keep only the BEST model:

In [ ]:
# Train with early stopping

torch.manual_seed(0)
model_es = SimpleNet().to(device)
optimizer_es = torch.optim.Adam(model_es.parameters(), lr=0.01)

best_val_loss = float('inf')
best_epoch = 0
patience = 10            # stop if no improvement for 10 epochs
patience_counter = 0
best_model_path = '/tmp/best_model.pth'

train_es, val_es = [], []

for epoch in range(200):  # plenty of epochs — early stop will catch us
    train_loss = train_one_epoch(model_es, train_loader, optimizer_es, loss_fn, device)
    val_loss, val_acc = evaluate(model_es, val_loader, loss_fn, device)
    
    train_es.append(train_loss)
    val_es.append(val_loss)
    
    # Check for improvement
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        # Save the BEST model
        torch.save(model_es.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n⏹  Early stopping at epoch {epoch}")
            print(f"   Best val loss was {best_val_loss:.4f} at epoch {best_epoch}")
            break

print(f"\nTotal epochs run: {len(train_es)} (would have run 200 without early stop)")

# Plot
plt.figure(figsize=(10, 5))
plt.plot(train_es, 'b-', label='Train loss', linewidth=2)
plt.plot(val_es, 'r-', label='Val loss', linewidth=2)
plt.axvline(x=best_epoch, color='green', linestyle='--', label=f'Best epoch ({best_epoch})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training with Early Stopping')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nThis pattern: train UNTIL val loss stops improving, then quit.")
print("Saves time AND prevents overfitting.")

## 8. The Reusable `Trainer` Class

Let's package EVERYTHING into a class we can use for any model:

In [ ]:
class Trainer:
    """A reusable training class with all the production features."""
    
    def __init__(self, model, optimizer, loss_fn, device='cpu',
                 scheduler=None, save_path='/tmp/best_model.pth'):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.device = device
        self.scheduler = scheduler
        self.save_path = save_path
        
        # History
        self.train_losses = []
        self.val_losses = []
        self.lrs = []
        self.best_val_loss = float('inf')
        self.best_epoch = 0
    
    def train_step(self, loader):
        """One epoch of training."""
        self.model.train()
        total_loss = 0
        n = 0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            logits = self.model(x)
            loss = self.loss_fn(logits, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item() * len(x)
            n += len(x)
        return total_loss / n
    
    def eval_step(self, loader):
        """One epoch of evaluation (no gradients)."""
        self.model.eval()
        total_loss = 0
        n = 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)
                logits = self.model(x)
                loss = self.loss_fn(logits, y)
                total_loss += loss.item() * len(x)
                n += len(x)
        return total_loss / n
    
    def fit(self, train_loader, val_loader, epochs=100, patience=10, verbose=True):
        """Full training loop with early stopping."""
        patience_counter = 0
        
        for epoch in range(epochs):
            # Track LR
            self.lrs.append(self.optimizer.param_groups[0]['lr'])
            
            # Train
            train_loss = self.train_step(train_loader)
            self.train_losses.append(train_loss)
            
            # Validate
            val_loss = self.eval_step(val_loader)
            self.val_losses.append(val_loss)
            
            # Step scheduler
            if self.scheduler:
                self.scheduler.step()
            
            # Track best + early stop
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.best_epoch = epoch
                patience_counter = 0
                torch.save(self.model.state_dict(), self.save_path)
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    if verbose:
                        print(f"⏹  Early stopping at epoch {epoch} (best epoch: {self.best_epoch})")
                    break
            
            if verbose and epoch % 10 == 0:
                print(f"Epoch {epoch:3d}: train={train_loss:.4f}, val={val_loss:.4f}, lr={self.lrs[-1]:.5f}")
    
    def load_best(self):
        """Load the best checkpoint."""
        self.model.load_state_dict(torch.load(self.save_path))
    
    def plot_history(self):
        """Plot loss curves and LR schedule."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
        
        ax1.plot(self.train_losses, 'b-', label='Train', linewidth=2)
        ax1.plot(self.val_losses, 'r-', label='Val', linewidth=2)
        ax1.axvline(x=self.best_epoch, color='green', linestyle='--', 
                    label=f'Best epoch ({self.best_epoch})')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        ax1.set_title('Training Progress')
        
        ax2.plot(self.lrs, 'm-', linewidth=2)
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Learning rate')
        ax2.set_title('LR Schedule')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()


# Use it!
torch.manual_seed(42)
model = SimpleNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
loss_fn = nn.BCEWithLogitsLoss()

trainer = Trainer(model, optimizer, loss_fn, device=device, scheduler=scheduler)
trainer.fit(train_loader, val_loader, epochs=100, patience=15)

trainer.plot_history()
trainer.load_best()
print(f"\nBest val loss: {trainer.best_val_loss:.4f} at epoch {trainer.best_epoch}")

---

## Exercises

1. **Try AdamW:** Replace `Adam` with `torch.optim.AdamW(..., weight_decay=0.01)`. Does it train better?

2. **Different schedulers:** Try `StepLR` with various step sizes. What happens with very aggressive (gamma=0.1) vs gentle (gamma=0.9) decay?

3. **Tiny batch size:** Set `batch_size=1` (this is true SGD!). Set `batch_size=400` (full-batch). How do the loss curves differ?

4. **Build a regression Trainer:** Modify the `Trainer` class to handle MSE regression instead of binary classification.

5. **Reload and continue:** Save a checkpoint mid-training, then load it into a fresh model and continue training. Does the loss pick up where it left off?

---

## Key Takeaways

### The 3 most important loss functions

| Task | Loss | PyTorch |
|------|------|---------|
| Predict number | MSE | `F.mse_loss` |
| Yes/no | BCE | `F.binary_cross_entropy_with_logits` |
| Pick one of K classes | CE | `F.cross_entropy` |

### The 3 most important optimizers

| Optimizer | Best For | Default LR |
|-----------|---------|-----------|
| SGD + momentum | Vision tasks (ResNet, etc.) | 0.01-0.1 |
| Adam | Default for most tasks | 0.001 |
| AdamW | Transformers, LLMs | 1e-4 to 1e-3 |

### The full training loop pattern

```python
for epoch in range(epochs):
    # Train
    model.train()
    for x, y in train_loader:
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Validate
    model.eval()
    with torch.no_grad():
        for x, y in val_loader:
            ...
    
    scheduler.step()                    # update LR
    
    if val_loss < best:                 # save best model
        torch.save(...)
```

### Things to remember

- Use **`with_logits`** loss versions when available (more stable)
- Always have **train/val** split — don't trust train loss alone
- **Mini-batches** > full-batch (memory + speed + better convergence)
- **`model.train()` / `model.eval()`** matters for dropout, batchnorm
- **Save the best model**, not the last one
- **Early stopping** prevents overfitting and saves time

**Tomorrow:** Day 09 — we'll properly use `nn.Module` and `nn.Sequential` to build clean, modular networks (and rebuild XOR with all of today's training infrastructure).